In [1]:
import random
import time

import numpy as np
import pandas as pd
import ptwt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    auc,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
    confusion_matrix
)
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchmetrics.classification import BinaryROC

/home/assalim/Desktop/Master/A-Wavelet-Based-Convolutional-Neural-Network-for-Deepfake-Audio-Detection/venv/lib/python3.13/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [2]:
EPOCHS = 100
BATCH_SIZE = 32
SEED = 101
WAVELET_FAMILIES = ["db4", "db6", "db8", "db14", "db16", "db18", "db24", "db26", "db28", "db34", "db36", "db38"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

cuda


In [3]:
def load_split(name):
    X = np.load(f"../../Data/{name}_features.npy", mmap_mode="r")
    meta = pd.read_csv(f"../../Data/{name}_meta.csv")

    X = np.asarray(X, dtype=np.float32).reshape(X.shape[0], -1, X.shape[-1])
    y = meta["is_bonafide"].to_numpy().astype(np.int64)

    return X, y, meta

In [4]:
t0 = time.time()
X_train, y_train, train_meta = load_split("train")
X_dev, y_dev, dev_meta = load_split("dev")
X_eval, y_eval, eval_meta = load_split("eval")
print(f"{time.time() - t0:.1f}s | train {X_train.shape} | dev {X_dev.shape} | eval {X_eval.shape}")

7.6s | train (25380, 40, 401) | dev (24844, 40, 401) | eval (71237, 40, 401)


In [5]:
t0 = time.time()
X_eval_2021, y_eval_2021, eval_meta_2021 = load_split("eval_2021")
print(f"{time.time() - t0:.1f}s | X_eval_2021 {X_eval_2021.shape}")

13.3s | X_eval_2021 (181566, 40, 401)


In [6]:
row_mean = X_train.mean(axis=(0, 2), keepdims=True)
row_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6

X_train = (X_train - row_mean) / row_std
X_dev = (X_dev - row_mean) / row_std
X_eval = (X_eval - row_mean) / row_std

In [7]:
X_eval_2021 = (X_eval_2021 - row_mean) / row_std

In [8]:
class AudioFeaturesDataset(Dataset):

    def __init__(self, X, y, train=False):
        self.X = torch.from_numpy(X).unsqueeze(1)
        self.y = torch.from_numpy(y)
        self.train = train
        self.mask = 2
        self.F = 0.25
        self.T = 0.15

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        if not self.train:
            return self.X[idx], self.y[idx]

        X = self.X[idx].clone()

        F = X.shape[1]  # frequency
        T = X.shape[2]  # time

        for _ in range(self.mask):
            # Frequency masking
            f = int(np.random.uniform(0, self.F * F))
            f0 = random.randint(0, F - f)
            X[:, f0:f0 + f, :] = 0

            # Time masking
            t = int(np.random.uniform(0, self.T * T))
            t0 = random.randint(0, T - t)
            X[:, :, t0:t0 + t] = 0
        
        return X, self.y[idx]

In [9]:
train_dataset = AudioFeaturesDataset(X_train, y_train, train=True)
dev_dataset = AudioFeaturesDataset(X_dev, y_dev)
eval_dataset = AudioFeaturesDataset(X_eval, y_eval)
eval_2021_dataset = AudioFeaturesDataset(X_eval_2021, y_eval_2021)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

eval_loader = DataLoader(
    eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

eval_2021_loader = DataLoader(
    eval_2021_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [10]:
class WaveletPooling(nn.Module):

    def __init__(self, family="db14", level=1):
        super().__init__()
        self.family = family
        self.level = level
        self.freq_pool = nn.AvgPool2d((2, 1), ceil_mode=True)

    def forward(self, x):
        if x.shape[-1] % 2:
            x = F.pad(x, (0, 1, 0, 0), mode="replicate")

        cA, cD = ptwt.wavedec(
            x,
            wavelet=self.family,
            level=self.level,
            mode="periodic"
        )

        return self.freq_pool(cA)

In [11]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )


class WaveletCNN(nn.Module):
    def __init__(self, wavelet_family):
        super().__init__()
        self.wavelet_family = wavelet_family

        self.features = nn.Sequential(
            conv_block(1, 32),
            conv_block(32, 32),
            WaveletPooling(self.wavelet_family),

            conv_block(32, 64),
            conv_block(64, 64),
            WaveletPooling(self.wavelet_family),

            conv_block(64, 128),
            conv_block(128, 128),
            WaveletPooling(self.wavelet_family),

            conv_block(128, 256),
            conv_block(256, 256),
            nn.Dropout2d(0.1),
        )

        self.classifier = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.mean(dim=2)
        x = torch.cat([x.mean(dim=2), x.std(dim=2)], dim=1)
        return self.classifier(x)

In [12]:
num_bonafide = int(y_train.sum())
num_spoof = int(len(y_train) - num_bonafide)

print(num_bonafide, num_spoof)

total = num_bonafide + num_spoof

weight_bonafide = total / (2 * num_bonafide)
weight_spoof = total / (2 * num_spoof)

class_weights = torch.tensor(
    [weight_spoof, weight_bonafide],
    dtype=torch.float32
).to(device)

2580 22800


In [13]:
class EarlyStopper:
    def __init__(self, patience=15, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best = float('inf')

    def early_stop(self, validation_metric):
        if validation_metric < self.best:
            self.best = validation_metric
            self.counter = 0
        else:
            self.counter += 1

        return self.counter >= self.patience

In [14]:
def gpu_calculate_eer(all_logits, all_labels, device):
    b_roc = BinaryROC().to(device)
    fpr, tpr, thresholds = b_roc(all_logits, all_labels)
    fnr = 1 - tpr
    idx = torch.argmin(torch.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2
    return eer.item(), thresholds[idx].item()


def llr_score(logits):
    return logits[:, 1] - logits[:, 0]

In [15]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    total = 0
    all_labels = []
    all_scores = []

    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        total += y.size(0)
        
        all_labels.append(y.clone())
        all_scores.append(llr_score(logits).clone())

    epoch_loss = running_loss / total
    
    epoch_eer, _ = gpu_calculate_eer(
        torch.cat(all_scores), 
        torch.cat(all_labels), 
        device
    )

    return epoch_loss, epoch_eer

In [16]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    total = 0

    all_labels = []
    all_scores = []

    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(X)
        loss = criterion(logits, y)
        running_loss += loss.item() * X.size(0)
        total += y.size(0)

        all_labels.append(y)
        all_scores.append(llr_score(logits))

    all_labels_gpu = torch.cat(all_labels)
    all_scores_gpu = torch.cat(all_scores)

    epoch_loss = running_loss / total
    
    eer, threshold = gpu_calculate_eer(all_scores_gpu, all_labels_gpu, device)
    
    all_predictions_gpu = (all_scores_gpu >= threshold).to(torch.int32)

    return (
        epoch_loss,
        eer,
        threshold,
        all_labels_gpu.cpu().numpy(),
        all_predictions_gpu.cpu().numpy(),
        all_scores_gpu.cpu().numpy(),
    )


In [17]:
def run_model(model, criterion, optimizer, scheduler):
    history = {
        "train_loss": [],
        "train_eer": [],
        "dev_loss": [],
        "dev_eer": [],
        "lr": [],
    }

    best_dev_eer = float("inf")
    best_model_state = None
    best_model_status = {}

    early_stopper = EarlyStopper(patience=10)
    for epoch in range(EPOCHS):
        train_loss, train_eer = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        dev_loss, dev_eer, _, _, _, _ = evaluate(
            model,
            dev_loader,
            criterion,
            device
        )

        scheduler.step(dev_eer)

        history["train_loss"].append(train_loss)
        history["train_eer"].append(train_eer)

        history["dev_loss"].append(dev_loss)
        history["dev_eer"].append(dev_eer)
        history["lr"].append(optimizer.param_groups[0]['lr'])

        if dev_eer < best_dev_eer:
            best_dev_eer = dev_eer
            best_model_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            best_model_status = {
                "epoch": epoch+1,
                "train_loss": train_loss,
                "train_eer": train_eer,
                "dev_loss": dev_loss,
                "best_dev_eer": best_dev_eer
            }

        if early_stopper.early_stop(dev_eer):
            print("Early Stopping...")
            break

        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"LR: {history['lr'][-1]:.6f} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train EER: {train_eer:.6f} | "
            f"Dev Loss: {dev_loss:.4f} | "
            f"Dev EER: {dev_eer:.6f} | "
            f"ES Counter: {early_stopper.counter}/{early_stopper.patience}"
        )

    print(f"Best Model: {best_model_status}")

    return history, best_model_state

In [18]:
def setup_model(wavelet_family):
    model = WaveletCNN(wavelet_family).to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.1,
        patience=5,
        threshold=1e-4,
        min_lr=1e-6
    )

    history, best_model_state = run_model(model, criterion, optimizer, scheduler)
    return model, criterion, history, best_model_state

In [19]:
def evaluate_asv_2019(model, eval_loader, criterion, device, wavelet_family):
    dev_loss, eer, _, eval_labels, _, eval_scores = evaluate(
        model,
        eval_loader,
        criterion,
        device
    )

    fpr, tpr, thresholds = roc_curve(
        eval_labels,
        eval_scores,
        pos_label=1
    )

    roc_auc = auc(fpr, tpr)


    return {
        "dataset": "ASVspoof 2019",
        "wavelet_family": wavelet_family,
        "eer": eer,
        "roc_auc": roc_auc,
    }

In [20]:
def evaluate_asv_2021(model, eval_loader, criterion, device, wavelet_family):
    dev_loss, eer, _, eval_labels, _, eval_scores = evaluate(
        model,
        eval_loader,
        criterion,
        device
    )

    fpr, tpr, thresholds = roc_curve(
        eval_labels,
        eval_scores,
        pos_label=1
    )

    roc_auc = auc(fpr, tpr)


    return {
        "dataset": "ASVspoof 2021",
        "wavelet_family": wavelet_family,
        "eer": eer,
        "roc_auc": roc_auc,
    }

In [21]:
results = []

for wavelet_family in WAVELET_FAMILIES:
    print(f"\n--- Processando Wavelet: {wavelet_family} ---")

    model, criterion, history, best_model_state = setup_model(wavelet_family)

    model.load_state_dict(best_model_state)
    model.to(device)
    model.eval()

    history_df = pd.DataFrame(history)
    history_df.to_csv(
        f"../../results/v2.5/wavelets/wavelet_results_v1.7/training_{wavelet_family}.csv",
        index=False
    )

    results.append(evaluate_asv_2019(model, eval_loader, criterion, device, wavelet_family))
    results.append(evaluate_asv_2021(model, eval_2021_loader, criterion, device, wavelet_family))
    

results_df = pd.DataFrame(results)

results_df.to_csv(
    "../../results/v2.5/wavelets/wavelet_results_v1.7/wavelet_results.csv",
    index=False
)

print("\nResultados salvos com sucesso!")
print(" - ../../results/v2.5/wavelets/wavelet_results_v1.7/wavelet_results.csv")
print(" - ../../results/v2.5/wavelets/wavelet_results_v1.7/training_<wavelet>.csv para cada wavelet")


--- Processando Wavelet: db4 ---
Epoch 01/100 | LR: 0.000100 | Train Loss: 0.2503 | Train EER: 0.103107 | Dev Loss: 0.0498 | Dev EER: 0.027859 | ES Counter: 0/10
Epoch 02/100 | LR: 0.000100 | Train Loss: 0.0970 | Train EER: 0.034507 | Dev Loss: 0.0350 | Dev EER: 0.010148 | ES Counter: 0/10
Epoch 03/100 | LR: 0.000100 | Train Loss: 0.0703 | Train EER: 0.025967 | Dev Loss: 0.0499 | Dev EER: 0.027769 | ES Counter: 1/10
Epoch 04/100 | LR: 0.000100 | Train Loss: 0.0531 | Train EER: 0.018209 | Dev Loss: 0.0267 | Dev EER: 0.004356 | ES Counter: 0/10
Epoch 05/100 | LR: 0.000100 | Train Loss: 0.0436 | Train EER: 0.015950 | Dev Loss: 0.0338 | Dev EER: 0.009817 | ES Counter: 1/10
Epoch 06/100 | LR: 0.000100 | Train Loss: 0.0380 | Train EER: 0.014342 | Dev Loss: 0.0543 | Dev EER: 0.005881 | ES Counter: 2/10
Epoch 07/100 | LR: 0.000100 | Train Loss: 0.0285 | Train EER: 0.008975 | Dev Loss: 0.0052 | Dev EER: 0.003538 | ES Counter: 0/10
Epoch 08/100 | LR: 0.000100 | Train Loss: 0.0332 | Train EER: 0